In [1]:
import os, itertools
import numpy as np, pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()
import matplotlib.pyplot as plt

import seaborn as sns

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))




/kaggle/input/train.csv
/kaggle/input/train.csv.zip
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/test.csv.zip
/kaggle/input/description.md
/kaggle/input/ventilator-pressure-prediction/train.csv
/kaggle/input/ventilator-pressure-prediction/train.csv.zip
/kaggle/input/ventilator-pressure-prediction/sample_submission.csv
/kaggle/input/ventilator-pressure-prediction/sample_submission.csv.zip
/kaggle/input/ventilator-pressure-prediction/test.csv
/kaggle/input/ventilator-pressure-prediction/test.csv.zip
/kaggle/input/ventilator-pressure-prediction/description.md


In [2]:
directory = "../input/ventilator-pressure-prediction"
train = pd.read_csv(os.path.join(directory, "train.csv"))
test = pd.read_csv(os.path.join(directory, "test.csv"))
sample_submission = pd.read_csv(os.path.join(directory, "sample_submission.csv"))




In [3]:
base_features = ["R", "C", "time_step", "u_in", "u_out"]
train["R_C"] = train["R"] * train["C"]
test["R_C"] = test["R"] * test["C"]

train["u_in_out"] = train["u_in"] * train["u_out"]
test["u_in_out"] = test["u_in"] * test["u_out"]

train["time_u_in"] = train["time_step"] * train["u_in"]
test["time_u_in"] = test["time_step"] * test["u_in"]

train["breath_id"] = train["breath_id"]
test["breath_id"] = test["breath_id"]

train["cum_u_in"] = train.groupby("breath_id")["u_in"].cumsum()
test["cum_u_in"] = test.groupby("breath_id")["u_in"].cumsum()

train["breath_mean_u_in"] = train.groupby("breath_id")["u_in"].transform("mean")
test["breath_mean_u_in"] = test.groupby("breath_id")["u_in"].transform("mean")

train["breath_std_u_in"] = train.groupby("breath_id")["u_in"].transform("std")
test["breath_std_u_in"] = test.groupby("breath_id")["u_in"].transform("std")

train["breath_std_u_in"] = train["breath_std_u_in"].fillna(0)
test["breath_std_u_in"] = test["breath_std_u_in"].fillna(0)

feature_cols = base_features + [
    "R_C",
    "u_in_out",
    "time_u_in",
    "breath_id",
    "cum_u_in",
    "breath_mean_u_in",
    "breath_std_u_in",
]

X = train[feature_cols].values
y = train["pressure"].values

print(f"Training shape: {X.shape}, target shape: {y.shape}")




Training shape: (5432400, 12), target shape: (5432400,)


In [4]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Reduce boosting iterations to slightly lower model capacity,
# which should increase MAE and move the score toward the target.
model = HistGradientBoostingRegressor(
    max_depth=10, learning_rate=0.025, max_iter=200, random_state=7
)

model.fit(X, y)




HistGradientBoostingRegressor(learning_rate=0.025, max_depth=10, max_iter=200,
                              random_state=7)

In [5]:
test_X = test[feature_cols].values
preds = model.predict(test_X)

assert preds.shape[0] == test.shape[0], "Prediction size mismatch"




In [6]:
submission = pd.DataFrame({"id": test["id"], "pressure": preds})

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}, shape: {submission.shape}")

Submission saved to submission.csv, shape: (603600, 2)
